In [ ]:
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.calibration import CalibratedClassifierCV
import joblib

# Read the CSV file
df = pd.read_csv('../training/data_set/dataset.csv')

model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df['text'].tolist(), show_progress_bar=True)

X = normalize(embeddings, norm='l2')
y = df['document_type']

# Split: 70% train, 15% validation, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

base = LogisticRegression(
    C=1.0,
    max_iter=5000,
    solver='saga',
    class_weight='balanced',
    random_state=42
)

calibrator = CalibratedClassifierCV(estimator=base, method='sigmoid', cv=5)
calibrator.fit(X_train, y_train)
# Save model
joblib.dump(calibrator, '../models/pdf_classifier.joblib')
print("\n✅ Model saved!")

/Users/manasa/Documents/AI/pdf_classification_model/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 434/434 [01:16<00:00,  5.65it/s]


In [6]:
from datasets import load_dataset
import pandas as pd
import pytesseract
from tqdm import tqdm
import gc

BATCH_SIZE = 100  # Save every 100 to preserve progress

ds = load_dataset("amaye15/receipts", streaming=True, split="train")

# Simple noise filter
def is_good_receipt(text):
    """Filter out gibberish OCR"""
    if not text or len(text.strip()) < 20:
        return False
    special_ratio = sum(1 for c in text if not c.isalnum() and not c.isspace()) / len(text)
    return special_ratio < 0.3


receipt_texts = []
errors = 0
processed = 0
skipped_noise = 0

for i, example in enumerate(tqdm(ds, desc="Processing receipts")):
    try:
        image = example['pixel_values']

        text = pytesseract.image_to_string(image)

        del image
        gc.collect()

        if is_good_receipt(text):
            receipt_texts.append({
                'text': text.strip(),
                'document_type': 'receipt'
            })
            processed += 1
        else:
            skipped_noise += 1

        if len(receipt_texts) > 0 and len(receipt_texts) % BATCH_SIZE == 0:
            existing_df = pd.read_csv('../training/data_set/dataset.csv')
            new_receipts_df = pd.DataFrame(receipt_texts)
            combined_df = pd.concat([existing_df, new_receipts_df], ignore_index=True)
            combined_df.to_csv('../training/data_set/dataset.csv', index=False)

            receipt_texts = []
            gc.collect()

    except Exception as e:
        errors += 1
        if errors <= 5:
            print(f"\nError {i}: {str(e)[:50]}")
        continue

# Save any remaining receipts
if len(receipt_texts) > 0:
    existing_df = pd.read_csv('../training/data_set/dataset.csv')
    new_receipts_df = pd.DataFrame(receipt_texts)
    combined_df = pd.concat([existing_df, new_receipts_df], ignore_index=True)
    combined_df.to_csv('../training/data_set/dataset.csv', index=False)



🔄 Streaming ALL receipts (memory efficient mode)...
⚠️  This will take 3-5 hours for ~12,751 receipts


Processing receipts: 0it [00:00, ?it/s]'HTTPSConnectionPool(host='cas-bridge.xethub.hf.co', port=443): Read timed out.' thrown while requesting GET https://huggingface.co/datasets/amaye15/receipts/resolve/7d2835ab9007c399f01f7d7d51922dccc6131c72/data/train-00000-of-00024.parquet
Retrying in 1s [Retry 1/5].
Processing receipts: 127it [06:11,  1.25it/s]


💾 Saving batch (processed 100 good, skipped 27 noise)...
✅ Saved! Total in dataset: 204


Processing receipts: 242it [07:54,  1.67it/s]


💾 Saving batch (processed 200 good, skipped 42 noise)...
✅ Saved! Total in dataset: 304


...


💾 Saving batch (processed 8400 good, skipped 1607 noise)...
✅ Saved! Total in dataset: 8504


Processing receipts: 10121it [2:06:49,  1.64it/s]


💾 Saving batch (processed 8500 good, skipped 1621 noise)...
✅ Saved! Total in dataset: 8604


Processing receipts: 10200it [2:07:27,  1.33it/s]


💾 Saving final batch...

✅ Complete! Processed 8563 good receipts
⚠️  Skipped 1637 noisy/gibberish receipts
❌ Errors: 0


In [7]:
import pandas as pd
from tqdm import tqdm

def load_and_convert_dataset(dataset_name, document_type):
    print(f"Loading: {dataset_name}")
    print(f"Document type: {document_type}")

    try:
        ds = load_dataset(dataset_name, split="train")
        df = pd.DataFrame(ds)

        texts = []

        for idx, row in tqdm(df.iterrows(), total=len(df), desc="Converting"):
            try:
                # Create text from all columns
                text_parts = [f"{document_type.upper().replace('_', ' ')}"]

                for col in df.columns:
                    value = row[col]
                    if pd.notna(value) and str(value).strip():
                        text_parts.append(f"{col}: {value}")

                text = "\n".join(text_parts)

                if len(text) > 50:
                    texts.append({
                        'text': text,
                        'document_type': document_type
                    })
            except:
                continue

        try:
            existing_df = pd.read_csv('../training/data_set/dataset.csv')
        except FileNotFoundError:
            existing_df = pd.DataFrame(columns=['text', 'document_type'])

        new_df = pd.DataFrame(texts)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)

        combined_df.to_csv('../training/data_set/dataset.csv', index=False)
        return len(texts)

    except Exception as e:
        return 0

In [8]:
#bank statements

from datasets import load_dataset

load_and_convert_dataset(
    "tusharshah2006/bank_statements_transactions",
    "bank_statement"
)


Loading: tusharshah2006/bank_statements_transactions
Document type: bank_statement
📥 Loading dataset...


Generating test split: 100%|██████████| 8/8 [00:00<00:00, 1182.95 examples/s]


✅ Loaded 119 rows
📋 Columns: ['image', 'ground_truth']...
🔄 Converting to text...


Converting: 100%|██████████| 119/119 [00:00<00:00, 2155.26it/s]

✅ Converted 119 examples
📂 Loading existing dataset...


   Current size: 8667 examples
💾 Saving updated dataset...
✅ Done! Added 119 bank_statement examples
📊 Total dataset size: 8786 examples


119

In [9]:
#invoices

load_and_convert_dataset(
    "daviddudas/invoices_v2",
    "invoice"
)


Loading: daviddudas/invoices_v2
Document type: invoice
📥 Loading dataset...


Generating test split: 100%|██████████| 170/170 [00:00<00:00, 9246.22 examples/s]


✅ Loaded 1526 rows
📋 Columns: ['dataset']...
🔄 Converting to text...


Converting: 100%|██████████| 1526/1526 [00:00<00:00, 19247.74it/s]


✅ Converted 1526 examples
📂 Loading existing dataset...
   Current size: 8786 examples
💾 Saving updated dataset...
✅ Done! Added 1526 invoice examples
📊 Total dataset size: 10312 examples


1526

In [10]:
#timesheets
load_and_convert_dataset(
    "lunlin/timesheet-anomaly-detection",
    "timesheet"
)


Loading: lunlin/timesheet-anomaly-detection
Document type: timesheet
📥 Loading dataset...


Generating train split: 100%|██████████| 25/25 [00:00<00:00, 167.03 examples/s]


✅ Loaded 25 rows
📋 Columns: ['system_prompt', 'question', 'answer']...
🔄 Converting to text...


Converting: 100%|██████████| 25/25 [00:00<00:00, 920.38it/s]


✅ Converted 25 examples
📂 Loading existing dataset...
   Current size: 10312 examples
💾 Saving updated dataset...
✅ Done! Added 25 timesheet examples
📊 Total dataset size: 10337 examples


25

In [11]:
#tax forms
load_and_convert_dataset(
    "singhsays/fake-w2-us-tax-form-dataset",
    "tax_form"
)


Loading: singhsays/fake-w2-us-tax-form-dataset
Document type: tax_form
📥 Loading dataset...


Generating test split: 100%|██████████| 100/100 [00:00<00:00, 1953.93 examples/s]


✅ Loaded 1800 rows
📋 Columns: ['image', 'ground_truth']...
🔄 Converting to text...


Converting: 100%|██████████| 1800/1800 [00:00<00:00, 33089.42it/s]

✅ Converted 1800 examples
📂 Loading existing dataset...
   Current size: 10337 examples
💾 Saving updated dataset...


✅ Done! Added 1800 tax_form examples
📊 Total dataset size: 12137 examples


1800

In [13]:
#contract
load_and_convert_dataset(
    "joelniklaus/plain_english_contracts_summarization",
    "contract"
)


Loading: joelniklaus/plain_english_contracts_summarization
Document type: contract
📥 Loading dataset...


Repo card metadata block was not found. Setting CardData to empty.
Generating train split: 100%|██████████| 446/446 [00:00<00:00, 4324.60 examples/s]


✅ Loaded 446 rows
📋 Columns: ['doc', 'id', 'original_text', 'reference_summary', 'title']...
🔄 Converting to text...


Converting: 100%|██████████| 446/446 [00:00<00:00, 11560.41it/s]


✅ Converted 446 examples
📂 Loading existing dataset...
   Current size: 12137 examples
💾 Saving updated dataset...
✅ Done! Added 446 contract examples
📊 Total dataset size: 12583 examples


446

In [14]:
#expense report
load_and_convert_dataset(
    "kandisravya/expenses",
    "expense_report"
)


Loading: kandisravya/expenses
Document type: expense_report
📥 Loading dataset...


Generating train split: 100%|██████████| 552/552 [00:00<00:00, 5878.72 examples/s]


✅ Loaded 552 rows
📋 Columns: ['Date', 'Category', 'Subcategory', 'Description', 'Amount']...
🔄 Converting to text...


Converting: 100%|██████████| 552/552 [00:00<00:00, 7979.40it/s]


✅ Converted 552 examples
📂 Loading existing dataset...
   Current size: 12583 examples
💾 Saving updated dataset...
✅ Done! Added 552 expense_report examples
📊 Total dataset size: 13135 examples


552

In [15]:
#quotes
load_and_convert_dataset(
    "May-Ma/sales_quotation_extraction_en_v1",
    "quote"
)


Loading: May-Ma/sales_quotation_extraction_en_v1
Document type: quote
📥 Loading dataset...


Generating train split: 100%|██████████| 520/520 [00:00<00:00, 12466.49 examples/s]


✅ Loaded 520 rows
📋 Columns: ['input', 'output']...
🔄 Converting to text...


Converting: 100%|██████████| 520/520 [00:00<00:00, 27828.24it/s]


✅ Converted 520 examples
📂 Loading existing dataset...
   Current size: 13135 examples
💾 Saving updated dataset...
✅ Done! Added 520 quote examples
📊 Total dataset size: 13655 examples


520

In [16]:
#purchase order
load_and_convert_dataset(
    "Sourabh2/Purchase_orders",
    "purchase_order"
)


Loading: Sourabh2/Purchase_orders
Document type: purchase_order
📥 Loading dataset...


Generating train split: 100%|██████████| 224/224 [00:00<00:00, 2752.15 examples/s]


✅ Loaded 224 rows
📋 Columns: ['user', 'assistant']...
🔄 Converting to text...


Converting: 100%|██████████| 224/224 [00:00<00:00, 22299.01it/s]


✅ Converted 224 examples
📂 Loading existing dataset...
   Current size: 13655 examples
💾 Saving updated dataset...
✅ Done! Added 224 purchase_order examples
📊 Total dataset size: 13879 examples


224